In [ ]:
!pip install kagglehub -Uq

In [ ]:
#!/bin/bash
# !curl -L -o "INPUT _DATA/fashion-product-images-small.zip" "https://www.kaggle.com/api/v1/datasets/download/paramaggarwal/fashion-product-images-small"
# !unzip -d  "INPUT_DATA" "/teamspace/studios/this_studio/INPUT_DATA/fashion-product-images-small.zip"

In [1]:
import os
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import json

# Define image transformation for CNN input
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Custom dataset class to load product images
class ProductDataset(Dataset):
    def __init__(self, product_ids, image_dir, transform=None):
        self.product_ids = product_ids
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.product_ids)

    def __getitem__(self, idx):
        img_id = self.product_ids[idx]
        img_path = os.path.join(self.image_dir, f"{img_id}.jpg")
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            # Handle missing images with a placeholder
            image = Image.new('RGB', (224, 224), color='gray')
        if self.transform:
            image = self.transform(image)
        return image, img_id

# Load styles.csv
styles = pd.read_csv('INPUT_DATA/styles.csv', on_bad_lines='skip')
product_ids = styles['id'].tolist()
image_dir = 'INPUT_DATA/images'

# Set up dataset and dataloader
dataset = ProductDataset(product_ids, image_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)

# Load pre-trained ResNet18 and remove the final layer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(pretrained=True)
feature_extractor = torch.nn.Sequential(*list(model.children())[:-1]).to(device)
feature_extractor.eval()

# Extract feature vectors
feature_dict = {}
with torch.no_grad():
    for images, ids in dataloader:
        images = images.to(device)
        features = feature_extractor(images)
        features = features.view(features.size(0), -1)  # Flatten to [batch_size, 512]
        for id_, feat in zip(ids, features):
            feature_dict[str(id_)] = feat.cpu()

# Save features
ids = list(feature_dict.keys())
features = torch.stack([feature_dict[id_] for id_ in ids])
torch.save(features, 'OUTPUT_DATA/features.pt')
with open('OUTPUT_DATA/ids.txt', 'w') as f:
    f.write('\n'.join(ids))

# Function to find similar products
def get_similar_products(target_id, features, ids, top_n=5):
    target_idx = ids.index(target_id)
    target_feat = features[target_idx]
    similarities = torch.nn.functional.cosine_similarity(target_feat.unsqueeze(0), features, dim=1)
    _, indices = torch.topk(similarities, top_n + 1)  # +1 to include self, then exclude
    similar_ids = [ids[i] for i in indices.tolist() if ids[i] != target_id][:top_n]
    return similar_ids

# Precompute similar products
similar_products = {id_: get_similar_products(id_, features, ids) for id_ in ids}

# Save similar products
with open('OUTPUT_DATA/similar_products.json', 'w') as f:
    json.dump(similar_products, f)

print("Preprocessing complete. Files saved: features.pt, ids.txt, similar_products.json")

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/zeus/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 162MB/s]


Preprocessing complete. Files saved: features.pt, ids.txt, similar_products.json


In [1]:
!zip  -r compressed_filename.zip OUTPUT_DATA

zsh:1: command not found: zip


In [2]:
import os
import shutil
# shutil.make_archive('output_file', 'zip', 'directory_to_zip')
shutil.make_archive('output_file', 'zip', 'OUTPUT_DATA')

'/teamspace/studios/this_studio/output_file.zip'

In [3]:
import os
import shutil
# shutil.make_archive('output_file', 'zip', 'directory_to_zip')
shutil.make_archive('input_file', 'zip', 'INPUT_DATA')

In [ ]:
import os
import shutil
# shutil.make_archive('output_file', 'zip', 'directory_to_zip')
shutil.make_archive('input_file', 'zip', 'INPUT_DATA')

In [1]:
import os
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import json

# Define image transformation for CNN input
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
])

# Custom dataset class to load product images
class ProductDataset(Dataset):
    def __init__(self, product_ids, image_dir, transform=None):
        self.product_ids = product_ids
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.product_ids)

    def __getitem__(self, idx):
        img_id = self.product_ids[idx]
        img_path = os.path.join(self.image_dir, f"{img_id}.jpg")
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            # Handle missing images with a placeholder image
            image = Image.new('RGB', (224, 224), color='gray')
        if self.transform:
            image = self.transform(image)
        return image, img_id

# Load styles.csv to get product IDs
styles = pd.read_csv('INPUT_DATA/styles.csv', on_bad_lines='skip')
product_ids = styles['id'].tolist()
image_dir = 'INPUT_DATA/images'

# Set up dataset and dataloader
dataset = ProductDataset(product_ids, image_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)

# Load pre-trained ResNet18 and remove the final classification layer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(pretrained=True)
feature_extractor = torch.nn.Sequential(*list(model.children())[:-1]).to(device)
feature_extractor.eval()

# Extract feature vectors and store in a dictionary
feature_dict = {}
with torch.no_grad():
    for images, ids in dataloader:
        images = images.to(device)
        features = feature_extractor(images)
        features = features.view(features.size(0), -1)  # Flatten to [batch_size, 512]
        for id_, feat in zip(ids, features):
            feature_dict[str(id_)] = feat.cpu()

# Save features
ids = list(feature_dict.keys())
features = torch.stack([feature_dict[id_] for id_ in ids])
os.makedirs('OUTPUT_DATA', exist_ok=True)
torch.save(features, 'OUTPUT_DATA/features_resnet50.pt')
with open('OUTPUT_DATA/ids_resnet50.txt', 'w') as f:
    f.write('\n'.join(ids))

# Function to find similar products using cosine similarity
def get_similar_products(target_id, features, ids, top_n=5):
    target_idx = ids.index(target_id)
    target_feat = features[target_idx]
    similarities = torch.nn.functional.cosine_similarity(target_feat.unsqueeze(0), features, dim=1)
    _, indices = torch.topk(similarities, top_n + 1)  # +1 to include the target itself
    similar_ids = [ids[i] for i in indices.tolist() if ids[i] != target_id][:top_n]
    return similar_ids

# Precompute similar products for each product ID
similar_products = {id_: get_similar_products(id_, features, ids) for id_ in ids}

# Save similar products mapping as JSON
with open('OUTPUT_DATA/similar_products_resnet50.json', 'w') as f:
    json.dump(similar_products, f)

print("Preprocessing complete. Files saved: features_resnet50.pt, ids_resnet50.txt, similar_products_resnet50.json")


/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/zeus/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 187MB/s]
